# Détection de profils atypiques Twitter - Approche non supervisée

## 1. Contexte et objectif

**Objectif :** appliquer K-Means et Isolation Forest pour détecter des profils atypiques.

**Fichier de données :** `users_labeled_manual.csv` (643 124 profils)

> **Même fichier que le supervisé**, mais les colonnes `label` et `anomaly_score` sont **exclues des features**.  
> Elles servent uniquement à **évaluer a posteriori** la concordance entre détection non supervisée et labellisation manuelle.

**Pipeline :** features (16 var.) → normalisation → **ACP (7 comp., ~79 % variance)** → K-Means / Isolation Forest

> **ACP non supervisée : 7 composantes** (seuil 75 %, aligné sur l'EDA section 11). Le supervisé utilise **5 composantes** (8 features) — choix indépendants.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_PATH = "users_labeled_manual.csv"
RANDOM_STATE = 42

In [ ]:
# Chargement — même fichier que le supervisé
df = pd.read_csv(DATA_PATH)

print("Shape :", df.shape)
print("\nColonnes :", df.columns.tolist())
print("\nLabels présents (non utilisés en entrée) :")
print(df["label"].value_counts())
print(f"Proportion atypiques : {df['label'].mean()*100:.1f} %")
print("\nAperçu :")
df.head()

In [ ]:
print(df.info())
print("\nStatistiques :")
df.describe()

## 2. Préparation des features

16 variables comportementales — **`label` et `anomaly_score` exclus** (réservés à l'évaluation).

In [ ]:
features = [
    "followers_count", "friends_count", "followers_friends_ratio",
    "nb_tweets", "nb_retweets", "retweet_ratio",
    "avg_tweet_length", "avg_hashtags", "avg_urls", "avg_mentions",
    "avg_favorites", "avg_retweet_count",
    "active_days", "tweet_frequency"
]

df["verified"] = df["verified"].astype(int)
df["default_profile_image"] = df["default_profile_image"].astype(int)
features += ["verified", "default_profile_image"]

# Exclusion explicite des colonnes de labellisation
assert "label" not in features
assert "anomaly_score" not in features

X = df[features].copy()
y_manual = df["label"].copy()  # conservé pour évaluation croisée uniquement

print("Features ML :", len(features))
print("Valeurs manquantes :", X.isnull().sum().sum())
print("Shape X :", X.shape)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Données normalisées — Shape :", X_scaled.shape)

## 3. Réduction dimensionnelle (ACP)

**Attention — ne pas confondre :**
| Choix | Objet | Méthode | Valeur retenue |
|-------|--------|---------|----------------|
| **Composantes ACP** | Réduction de dimensions | Seuil de variance cumulée (70–80 %) | **7** (~79 %) |
| **Clusters K-Means (k)** | Regroupement | Coude sur l'inertie (section 4) | **7** (indépendant de l'ACP) |

Le **7 de l'ACP** et le **k=7 du K-Means** sont deux décisions **distinctes** (coïncidence numérique seulement).

In [ ]:
pca_full = PCA()
pca_full.fit(X_scaled)

evr = pca_full.explained_variance_ratio_
variance_cumulee = np.cumsum(evr) * 100
eigenvalues = pca_full.explained_variance_

# --- Critères de sélection du nombre de composantes ---
n_kaiser = int(np.sum(eigenvalues > 1))
n_70 = int(np.searchsorted(variance_cumulee, 70) + 1)
n_75 = int(np.searchsorted(variance_cumulee, 75) + 1)
n_80 = int(np.searchsorted(variance_cumulee, 80) + 1)

# Retenu : premier n tel que variance cumulée >= 75 % (compromis entre seuils 70 % et 80 %)
n_components = n_75

print("=" * 60)
print(" SÉLECTION DU NOMBRE DE COMPOSANTES ACP")
print("=" * 60)
print(f"Critère Kaiser (valeur propre > 1)     : {n_kaiser} composantes")
print(f"Seuil 70 % variance cumulée            : {n_70} composantes ({variance_cumulee[n_70-1]:.1f} %)")
print(f"Seuil 75 % variance cumulée (retenu)   : {n_components} composantes ({variance_cumulee[n_components-1]:.1f} %)")
print(f"Seuil 80 % variance cumulée            : {n_80} composantes ({variance_cumulee[n_80-1]:.1f} %)")
print(f"\n=> n_components retenu = {n_components}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, len(evr) + 1), evr * 100, "bo-")
axes[0].set_xlabel("Composante")
axes[0].set_ylabel("Variance individuelle (%)")
axes[0].set_title("Scree plot ACP")
axes[0].grid(True)

axes[1].plot(range(1, len(variance_cumulee) + 1), variance_cumulee, "bo-")
axes[1].axhline(y=70, color="r", linestyle="--", label="70 %")
axes[1].axhline(y=80, color="g", linestyle="--", label="80 %")
axes[1].axvline(x=n_components, color="orange", linestyle=":", linewidth=2,
                label=f"Retenu = {n_components} ({variance_cumulee[n_components-1]:.1f} %)")
axes[1].set_xlabel("Nombre de composantes")
axes[1].set_ylabel("Variance cumulée (%)")
axes[1].set_title("Variance cumulée — seuil 75 %")
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()

for i in range(min(10, len(variance_cumulee))):
    print(f"  PC{i+1}: {evr[i]*100:.2f} % | cumulé: {variance_cumulee[i]:.2f} %")

In [ ]:
pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print(f"\nShape après ACP : {X_pca.shape}")
print(f"Variance expliquée totale : {sum(pca.explained_variance_ratio_)*100:.2f} %")

## 4. Modèle 1 — K-Means

On utilise `MiniBatchKMeans` (adapté aux grands volumes).

**Méthode du coude ici = nombre de clusters k** (pas le nombre de composantes ACP).
La chute d'inertie la plus marquée se produit entre **k=6 et k=7** → **k=7** retenu.

In [ ]:
inertia = []
K_range = range(2, 11)

# Coude K-Means : choix du nombre de clusters k (pas composantes ACP)
print("Calcul en cours... (peut prendre 1-2 minutes)")

for k in K_range:
    kmeans_tmp = MiniBatchKMeans(
        n_clusters=k, random_state=RANDOM_STATE,
        batch_size=10000, n_init=3
    )
    kmeans_tmp.fit(X_pca)
    inertia.append(kmeans_tmp.inertia_)
    print(f"  k={k} → inertie={kmeans_tmp.inertia_:.0f}")

plt.figure(figsize=(10, 5))
plt.plot(K_range, inertia, "bo-", linewidth=2, markersize=8)
plt.xlabel("Nombre de clusters (k)")
plt.ylabel("Inertie")
plt.title("K-Means — Méthode du coude")
plt.grid(True)
plt.xticks(K_range)
plt.show()

In [ ]:
# k retenu via coude sur l'inertie (cellule precedente)
# Indépendant du nombre de composantes ACP (7)
k_optimal = 7

kmeans = MiniBatchKMeans(
    n_clusters=k_optimal, random_state=RANDOM_STATE,
    batch_size=10000, n_init=10
)
kmeans_labels = kmeans.fit_predict(X_pca)
df["cluster_kmeans"] = kmeans_labels

cluster_counts = df["cluster_kmeans"].value_counts().sort_index()
print("Distribution des clusters K-Means :")
print(cluster_counts)

plt.figure(figsize=(8, 5))
cluster_counts.plot(kind="bar", color="steelblue", edgecolor="black")
plt.title(f"Distribution des clusters — K-Means (k={k_optimal})")
plt.xlabel("Cluster")
plt.ylabel("Nombre de profils")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

threshold = len(df) * 0.01
atypiques_kmeans = df[df["cluster_kmeans"].isin(
    cluster_counts[cluster_counts < threshold].index
)]
print(f"\nProfils atypiques K-Means (clusters < 1 %) : {len(atypiques_kmeans):,}")
print(f"Soit {len(atypiques_kmeans)/len(df)*100:.2f} % du dataset")

## 5. Modèle 2 — Isolation Forest

Méthode conçue spécifiquement pour la détection d'anomalies.  
Labels internes : **-1** = anomalie (atypique), **+1** = normal.

In [ ]:
print("Isolation Forest en cours... ")

iso_forest = IsolationForest(
    n_estimators=100,
    contamination='auto',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

iso_labels = iso_forest.fit_predict(X_pca)
df["anomalie_isoforest"] = iso_labels

n_anomalies = (iso_labels == -1).sum()
n_normaux = (iso_labels == 1).sum()

print(f"\nRésultats Isolation Forest :")
print(f"  Profils normaux   : {n_normaux:,}  ({n_normaux/len(df)*100:.2f} %)")
print(f"  Profils atypiques : {n_anomalies:,} ({n_anomalies/len(df)*100:.2f} %)")

labels_plot = ["Normaux", "Atypiques"]
values_plot = [n_normaux, n_anomalies]
colors = ["steelblue", "crimson"]

plt.figure(figsize=(7, 5))
plt.bar(labels_plot, values_plot, color=colors, edgecolor="black")
plt.title("Isolation Forest — Détection de profils atypiques")
plt.ylabel("Nombre de profils")
plt.grid(axis="y")
for i, v in enumerate(values_plot):
    plt.text(i, v + 1000, f"{v:,}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Comparaison des deux méthodes

In [ ]:
cluster_counts = df["cluster_kmeans"].value_counts().sort_index()
threshold = len(df) * 0.01

df["atypique_kmeans"] = df["cluster_kmeans"].isin(
    cluster_counts[cluster_counts < threshold].index
).astype(int)

df["atypique_isoforest"] = (df["anomalie_isoforest"] == -1).astype(int)

print("=" * 50)
print(" RÉSUMÉ COMPARATIF")
print("=" * 50)
print(f"K-Means    — Atypiques : {df['atypique_kmeans'].sum():>7,}  "
      f"({df['atypique_kmeans'].mean()*100:.2f} %)")
print(f"Iso Forest — Atypiques : {df['atypique_isoforest'].sum():>7,}  "
      f"({df['atypique_isoforest'].mean()*100:.2f} %)")

accord = (df["atypique_kmeans"] == df["atypique_isoforest"]).sum()
print(f"\nAccord global          : {accord:,} profils ({accord/len(df)*100:.2f} %)")

les_deux = df[(df["atypique_kmeans"] == 1) & (df["atypique_isoforest"] == 1)]
kmeans_seul = df[(df["atypique_kmeans"] == 1) & (df["atypique_isoforest"] == 0)]
iso_seul = df[(df["atypique_kmeans"] == 0) & (df["atypique_isoforest"] == 1)]

print(f"Atypiques par LES DEUX : {len(les_deux):,} profils")
print(f"K-Means seul           : {len(kmeans_seul):,} profils")
print(f"Iso Forest seul        : {len(iso_seul):,} profils")

In [ ]:
categories = ["K-Means\nseulement", "Les DEUX\nméthodes", "IsoForest\nseulement"]
values = [len(kmeans_seul), len(les_deux), len(iso_seul)]
colors = ["#2196F3", "#9C27B0", "#F44336"]

plt.figure(figsize=(9, 5))
bars = plt.bar(categories, values, color=colors, edgecolor="black", width=0.5)
plt.title("Comparaison K-Means vs Isolation Forest\n(Profils atypiques détectés)",
          fontsize=13, fontweight="bold")
plt.ylabel("Nombre de profils")
plt.grid(axis="y", alpha=0.4)
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f"{val:,}\n({val/len(df)*100:.2f} %)",
             ha="center", fontweight="bold", fontsize=10)
plt.tight_layout()
plt.show()

cm = confusion_matrix(df["atypique_isoforest"], df["atypique_kmeans"])
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt=",", cmap="Blues",
            xticklabels=["K-Means: Normal", "K-Means: Atypique"],
            yticklabels=["IsoForest: Normal", "IsoForest: Atypique"])
plt.title("Matrice de concordance — K-Means vs Isolation Forest", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
cols_analyse = [
    "followers_count", "friends_count", "nb_tweets",
    "retweet_ratio", "avg_hashtags", "avg_mentions",
    "tweet_frequency", "followers_friends_ratio"
]

normaux = df[(df["atypique_kmeans"] == 0) & (df["atypique_isoforest"] == 0)]

comparison = pd.DataFrame({
    "Normaux": normaux[cols_analyse].mean(),
    "Atypiques (K-Means)": df[df["atypique_kmeans"] == 1][cols_analyse].mean(),
    "Atypiques (IsoForest)": df[df["atypique_isoforest"] == 1][cols_analyse].mean(),
    "Atypiques (Les deux)": les_deux[cols_analyse].mean()
})

print("PROFIL MOYEN — Atypiques vs normaux")
print("=" * 60)
print(comparison.round(2).to_string())

## 7. Évaluation croisée vs labels manuels

Grâce au fichier commun `users_labeled_manual.csv`, on compare les détections non supervisées aux **labels manuels Excel** (sans les avoir utilisés en entrée).

In [ ]:
def eval_vs_manual(y_pred, name):
    f1 = f1_score(y_manual, y_pred)
    prec = precision_score(y_manual, y_pred)
    rec = recall_score(y_manual, y_pred)
    print(f"\n{name} vs labels manuels :")
    print(f"  F1={f1:.4f}  Precision={prec:.4f}  Recall={rec:.4f}")
    return f1, prec, rec

eval_vs_manual(df["atypique_kmeans"], "K-Means")
eval_vs_manual(df["atypique_isoforest"], "Isolation Forest")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pred_col, title in zip(
    axes,
    ["atypique_kmeans", "atypique_isoforest"],
    ["K-Means vs labels manuels", "Isolation Forest vs labels manuels"]
):
    cm = confusion_matrix(y_manual, df[pred_col])
    sns.heatmap(cm, annot=True, fmt=",", cmap="Blues", ax=ax,
                xticklabels=["Prédit: Normal", "Prédit: Atypique"],
                yticklabels=["Label: Normal", "Label: Atypique"])
    ax.set_title(title, fontweight="bold")

plt.tight_layout()
plt.show()

## 8. Méthode retenue

| Critère | K-Means | Isolation Forest |
|---------|---------|------------------|
| Sensibilité | Faible (~0,5 %) | Élevée (~6,7 %) |
| Précision | Très élevée (noyau dur) | Bonne |
| Type d'anomalie | Clusters minoritaires | Profils déviants multidimensionnels |
| Interprétabilité | Facile (par cluster) | Score d'anomalie abstrait |
| Adapté à la détection d'anomalies | Indirectement | **Oui, conçu pour cela** |

**Isolation Forest est retenu** car :
- il est spécifiquement conçu pour la détection d'anomalies ;
- il capture ~10× plus de profils suspects que K-Means ;
- il détecte des anomalies diffuses (ratios extrêmes, audiences gonflées) que K-Means ne voit pas.

**K-Means reste complémentaire** : il identifie un noyau dur de profils fortement atypiques, entièrement confirmé par Isolation Forest.

---

> L'application conjointe de K-Means (*k = 7*) et d'Isolation Forest sur **643 124 profils Twitter**, après ACP à **7 composantes**, met en évidence deux niveaux de détection. K-Means isole un noyau compact de profils extrêmes ; Isolation Forest étend la détection à ~6,7 % du dataset (contamination = auto). La convergence des deux méthodes sur ce noyau commun constitue un signal robuste de comportements anormaux (bots, comptes coordonnés, audiences artificielles).

In [ ]:
# Sélection automatique de la méthode retenue
nb_kmeans = df["atypique_kmeans"].sum()
nb_iso = df["atypique_isoforest"].sum()
nb_consensus = len(les_deux)

print("=" * 60)
print(" MÉTHODE RETENUE : Isolation Forest")
print("=" * 60)
print(f"\nK-Means       : {nb_kmeans:,} atypiques ({nb_kmeans/len(df)*100:.2f} %)")
print(f"Iso Forest    : {nb_iso:,} atypiques ({nb_iso/len(df)*100:.2f} %)")
print(f"Consensus     : {nb_consensus:,} profils ({nb_consensus/len(df)*100:.2f} %)")
print(f"\nK-Means seul  : {len(kmeans_seul):,} (0 % des atypiques K-Means hors consensus)")
print(f"Iso Forest seul : {len(iso_seul):,} profils supplémentaires détectés")